In [1]:
import argparse
import time

import pandas as pd
import xarray as xr
from argopy import ArgoIndex
import zarr


/runtime/libs/client-lib/.venv/bin/python: No module named pip


In [2]:
# argo positions 
idx = ArgoIndex(index_file="core", cache=False)
idx.query.wmo(6900166)
cols = ["wmo", "cyc", "date", "latitude", "longitude"]
argo_df = idx.to_dataframe()[cols].sort_values("cyc").dropna().reset_index(drop=True)
idx

<argoindex.pandas>
Host: https://data-argo.ifremer.fr
Index: ar_index_global_prof.txt.gz
Convention: ar_index_global_prof (Profile directory file of the Argo GDAC)
In memory: True (3425928 records)
Searched: True (124 matches, 0.0036%) - {'WMO': [6900166]}

In [3]:
print(argo_df["date"].min().floor("D"), argo_df["date"].max().ceil("D"))

2002-05-16 00:00:00 2005-09-28 00:00:00


In [ ]:
zarr_url = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"

variables = [
    "sea_surface_temperature",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    
    "surface_latent_heat_flux",
    "surface_sensible_heat_flux",
    "surface_net_solar_radiation",
    "surface_net_thermal_radiation",
]


ds = xr.open_zarr(zarr_url, consolidated=True, chunks=None,
                      storage_options={"token": "anon"})[variables]

In [5]:
print(ds.sel(time=slice(argo_df["date"].min().floor("D"), argo_df["date"].max().ceil("D"))))


<xarray.Dataset> Size: 859GB
Dimensions:                        (time: 29545, latitude: 721, longitude: 1440)
Coordinates:
  * time                           (time) datetime64[ns] 236kB 2002-05-16 ......
  * latitude                       (latitude) float32 3kB 90.0 89.75 ... -90.0
  * longitude                      (longitude) float32 6kB 0.0 0.25 ... 359.8
Data variables:
    sea_surface_temperature        (time, latitude, longitude) float32 123GB ...
    10m_u_component_of_wind        (time, latitude, longitude) float32 123GB ...
    10m_v_component_of_wind        (time, latitude, longitude) float32 123GB ...
    surface_latent_heat_flux       (time, latitude, longitude) float32 123GB ...
    surface_sensible_heat_flux     (time, latitude, longitude) float32 123GB ...
    surface_net_solar_radiation    (time, latitude, longitude) float32 123GB ...
    surface_net_thermal_radiation  (time, latitude, longitude) float32 123GB ...
Attributes:
    last_updated:           2026-10-22 05:51

In [ ]:

from floatmatcher.product import ERA5Product, to_standard
import floatmatcher.floatmatcher as fm
import numpy as np 

my_fm = fm.FloatMatcher()
my_fm.set_points(argo_df, lon="longitude", lat="latitude", time="date")
my_fm.set_method("nearest", radius=25, time_gap=np.timedelta64(1, "h"))


# select only houres before and after ach profile. all hours on this time period are exceeding dask graph resolver...
hours = np.unique(np.concatenate([argo_df["date"].dt.floor("h"),
                                  argo_df["date"].dt.ceil("h")]))
ds_std = to_standard(ds.sel(time=hours), ERA5Product.coord_map)  



In [14]:
print(np.shape(hours))

(248,)


In [9]:
my_fm._result = my_fm.method.apply(ds_std, my_fm.points)   # à la place de match()
my_fm.reinject()

,wmo,cyc,date,latitude,longitude,sea_surface_temperature_coloc,10m_u_component_of_wind_coloc,10m_v_component_of_wind_coloc,surface_latent_heat_flux_coloc,surface_sensible_heat_flux_coloc,surface_net_solar_radiation_coloc,surface_net_thermal_radiation_coloc
0,6900166,1,2002-05-16 17:21:59,39.640,-17.036,289.253998,9.126350,0.712574,-420866.9375,-40671.87500,1.598822e+06,-276125.56250
1,6900166,2,2002-05-26 17:45:30,39.733,-16.888,289.680420,4.305833,-5.648382,-273319.5625,-18209.12500,9.379833e+05,-118308.68750
2,6900166,3,2002-06-05 17:54:51,39.714,-16.774,290.221069,2.779732,-5.828719,-267390.0625,-20134.93750,1.152761e+06,-135145.59375
3,6900166,4,2002-06-15 18:19:34,39.572,-16.884,292.115631,4.679074,4.845134,-78630.2500,18786.06250,1.594572e+06,-262017.31250
4,6900166,5,2002-06-25 20:07:54,39.444,-16.170,292.345764,-4.931728,-11.709999,-876229.0000,-119863.00000,2.520592e+05,-177786.53125
...,...,...,...,...,...,...,...,...,...,...,...,...
119,6900166,120,2005-08-18 21:32:35,36.566,-32.206,298.299072,-1.765296,0.948642,-147182.5625,-6928.37500,1.250000e-01,-209358.15625
120,6900166,121,2005-08-28 18:20:14,36.490,-32.307,298.103851,1.698326,-3.357248,-319359.6250,-32039.37500,1.829372e+06,-269789.93750
121,6900166,122,2005-09-07 21:13:50,36.473,-32.749,297.525116,4.991060,-0.818947,-304905.4375,-6360.93750,2.881838e+04,-245789.50000
122,6900166,123,2005-09-17 17:57:57,36.525,-32.948,297.714294,2.152675,-1.838766,-156472.4375,-19740.62500,1.375876e+06,-209355.59375
